# PVD公开实验数据：第一轮质量检查

本Notebook读取30条已核实的PVD/磁控溅射实验记录，并自动检查数据结构、完整性、唯一性、数值类型、基本物理合理性以及来源状态。

**分析原则：**这里只读取主数据CSV。9条存疑记录保留在附录中，不参与本次检查和后续建模。

In [1]:
from pathlib import Path
import json
import re
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 120)

DATA_FILENAME = 'pvd_experiments_verified_v3.csv'
EXPECTED_ROWS = 30

def locate_data_file(filename=DATA_FILENAME):
    start = Path.cwd().resolve()
    candidates = []
    for root in [start, *start.parents]:
        candidates.extend([
            root / filename,
            root / 'outputs' / 'pvd_public_data_v3' / filename,
            root / '项目名' / 'outputs' / 'pvd_public_data_v3' / filename,
        ])
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f'未找到 {filename}，请确认Notebook位于项目目录中。')

data_path = locate_data_file()
output_dir = data_path.parent / 'quality_checks'
output_dir.mkdir(parents=True, exist_ok=True)
print('数据文件：', data_path)

数据文件： /mnt/d/学业/项目名/outputs/pvd_public_data_v3/pvd_experiments_verified_v3.csv


## 1. 读取并预览主数据

In [2]:
df = pd.read_csv(data_path)
print(f'数据规模：{df.shape[0]} 行 × {df.shape[1]} 列')
display(df.head(5))

数据规模：30 行 × 22 列


,experiment_id,source_id,source_locator,target_material,deposition_method,deposition_time_min,pressure_Pa,film_thickness_nm,power_W,discharge_current_mA,substrate,substrate_temperature_C,substrate_temperature_label,deposition_rate_nm_min,original_deposition_time_s,source_pressure_original,source_thickness_original,source_doi,source_url,thickness_status,verification_status,comments
0,CU3N-01,S02,Table 1 - 3.5 Pa - 25 W,Cu3N,reactive RF magnetron sputtering,30,3.5,45.0,25.0,NaN,glass,NaN,RT,NaN,1800.0,3.5 Pa,45 nm,10.3390/ma16041508,https://pmc.ncbi.nlm.nih.gov/articles/PMC9965105/,reported_measured,verified,Thickness measured by stylus profilometry; pure nitrogen process gas
1,CU3N-02,S02,Table 1 - 3.5 Pa - 50 W,Cu3N,reactive RF magnetron sputtering,30,3.5,85.0,50.0,NaN,glass,NaN,RT,NaN,1800.0,3.5 Pa,85 nm,10.3390/ma16041508,https://pmc.ncbi.nlm.nih.gov/articles/PMC9965105/,reported_measured,verified,Thickness measured by stylus profilometry; pure nitrogen process gas
2,CU3N-03,S02,Table 1 - 3.5 Pa - 75 W,Cu3N,reactive RF magnetron sputtering,15,3.5,95.0,75.0,NaN,glass,NaN,RT,NaN,900.0,3.5 Pa,95 nm,10.3390/ma16041508,https://pmc.ncbi.nlm.nih.gov/articles/PMC9965105/,reported_measured,verified,Thickness measured by stylus profilometry; pure nitrogen process gas
3,CU3N-04,S02,Table 1 - 3.5 Pa - 100 W,Cu3N,reactive RF magnetron sputtering,15,3.5,100.0,100.0,NaN,glass,NaN,RT,NaN,900.0,3.5 Pa,100 nm,10.3390/ma16041508,https://pmc.ncbi.nlm.nih.gov/articles/PMC9965105/,reported_measured,verified,Thickness measured by stylus profilometry; pure nitrogen process gas
4,CU3N-05,S02,Table 1 - 3.5 Pa - 150 W,Cu3N,reactive RF magnetron sputtering,10,3.5,175.0,150.0,NaN,glass,NaN,RT,NaN,600.0,3.5 Pa,175 nm,10.3390/ma16041508,https://pmc.ncbi.nlm.nih.gov/articles/PMC9965105/,reported_measured,verified,Thickness measured by stylus profilometry; pure nitrogen process gas


## 2. 自动质量检查

功率和放电电流属于互补字段：每条记录至少需要其中一个有效值。温度、沉积速率等字段允许缺失，不会被误判为错误。

In [3]:
required_columns = [
    'experiment_id', 'source_id', 'source_locator', 'target_material',
    'deposition_method', 'deposition_time_min', 'pressure_Pa',
    'film_thickness_nm', 'substrate', 'source_doi', 'source_url',
    'thickness_status', 'verification_status'
]
numeric_columns = [
    'deposition_time_min', 'pressure_Pa', 'film_thickness_nm', 'power_W',
    'discharge_current_mA', 'substrate_temperature_C',
    'deposition_rate_nm_min', 'original_deposition_time_s'
]
optional_columns = [
    'power_W', 'discharge_current_mA', 'substrate_temperature_C',
    'substrate_temperature_label', 'deposition_rate_nm_min',
    'original_deposition_time_s', 'comments'
]

checks = []
issues = []

def add_check(name, passed, observed, expected, details):
    checks.append({
        'check': name,
        'result': 'PASS' if passed else 'FAIL',
        'observed': observed,
        'expected': expected,
        'details': details,
    })

add_check('row_count', len(df) == EXPECTED_ROWS, len(df), EXPECTED_ROWS,
          '主数据表应包含30条已核实实验记录。')

missing_columns = [c for c in required_columns if c not in df.columns]
add_check('required_columns_present', not missing_columns,
          len(required_columns) - len(missing_columns), len(required_columns),
          '缺失列：' + (', '.join(missing_columns) if missing_columns else '无'))

if missing_columns:
    raise ValueError('缺少必填列，停止后续检查：' + ', '.join(missing_columns))

id_text = df['experiment_id'].astype('string').str.strip()
missing_id = id_text.isna() | id_text.eq('')
duplicate_id = id_text.duplicated(keep=False) & ~missing_id
add_check('experiment_id_complete', not missing_id.any(), int(missing_id.sum()), 0,
          '每条记录必须有实验编号。')
add_check('experiment_id_unique', not duplicate_id.any(), int(duplicate_id.sum()), 0,
          '实验编号必须唯一。')

for idx in df.index[missing_id]:
    issues.append({'row': int(idx)+2, 'experiment_id': '', 'check': 'missing_experiment_id', 'details': '实验编号为空'})
for idx in df.index[duplicate_id]:
    issues.append({'row': int(idx)+2, 'experiment_id': df.at[idx, 'experiment_id'], 'check': 'duplicate_experiment_id', 'details': '实验编号重复'})

missing_required = df[required_columns].isna() | df[required_columns].astype('string').apply(lambda s: s.str.strip().eq(''))
missing_required_count = int(missing_required.to_numpy().sum())
add_check('required_values_complete', missing_required_count == 0, missing_required_count, 0,
          '必填字段不得为空。')
for row_pos, col_pos in zip(*missing_required.to_numpy().nonzero()):
    issues.append({
        'row': int(row_pos)+2,
        'experiment_id': df.iloc[row_pos]['experiment_id'],
        'check': 'missing_required_value',
        'details': f'{required_columns[col_pos]} 为空',
    })

numeric_data = {}
numeric_failure_count = 0
for column in numeric_columns:
    if column not in df.columns:
        continue
    converted = pd.to_numeric(df[column], errors='coerce')
    numeric_data[column] = converted
    invalid = df[column].notna() & converted.isna()
    numeric_failure_count += int(invalid.sum())
    for idx in df.index[invalid]:
        issues.append({'row': int(idx)+2, 'experiment_id': df.at[idx, 'experiment_id'], 'check': 'non_numeric_value', 'details': f'{column} 不是有效数值'})
add_check('numeric_fields_parseable', numeric_failure_count == 0, numeric_failure_count, 0,
          '已有的数值字段必须能够转换为数值。')

nonpositive_count = 0
for column in ['deposition_time_min', 'pressure_Pa', 'film_thickness_nm']:
    invalid = numeric_data[column].notna() & numeric_data[column].le(0)
    nonpositive_count += int(invalid.sum())
    for idx in df.index[invalid]:
        issues.append({'row': int(idx)+2, 'experiment_id': df.at[idx, 'experiment_id'], 'check': 'nonpositive_physical_value', 'details': f'{column} 必须大于0'})
add_check('core_physical_values_positive', nonpositive_count == 0, nonpositive_count, 0,
          '沉积时间、压力和膜厚必须大于0。')

power = numeric_data.get('power_W', pd.Series(index=df.index, dtype=float))
current = numeric_data.get('discharge_current_mA', pd.Series(index=df.index, dtype=float))
bad_electrical = (power.isna() & current.isna()) | (power.notna() & power.le(0)) | (current.notna() & current.le(0))
add_check('power_or_current_available', not bad_electrical.any(), int(bad_electrical.sum()), 0,
          '每条记录至少需要一个正的功率或放电电流值。')
for idx in df.index[bad_electrical]:
    issues.append({'row': int(idx)+2, 'experiment_id': df.at[idx, 'experiment_id'], 'check': 'electrical_parameter', 'details': '功率和电流均缺失，或存在非正值'})

doi_pattern = re.compile(r'^10\.\d{4,9}/\S+$')
doi_text = df['source_doi'].astype('string').str.strip()
url_text = df['source_url'].astype('string').str.strip()
bad_source = (~doi_text.fillna('').map(lambda x: bool(doi_pattern.match(x)))) | (~url_text.fillna('').str.startswith(('https://', 'http://')))
add_check('source_identifiers_valid', not bad_source.any(), int(bad_source.sum()), 0,
          'DOI格式和来源URL必须可识别。')
for idx in df.index[bad_source]:
    issues.append({'row': int(idx)+2, 'experiment_id': df.at[idx, 'experiment_id'], 'check': 'source_identifier', 'details': 'DOI或URL格式无效'})

bad_status = (df['verification_status'].astype('string').str.strip() != 'verified') | (df['thickness_status'].astype('string').str.strip() != 'reported_measured')
add_check('verification_status_consistent', not bad_status.any(), int(bad_status.sum()), 0,
          '主表仅允许 verified 且 reported_measured 的记录。')
for idx in df.index[bad_status]:
    issues.append({'row': int(idx)+2, 'experiment_id': df.at[idx, 'experiment_id'], 'check': 'verification_status', 'details': '记录状态不符合主表规则'})

quality_summary = pd.DataFrame(checks)
quality_issues = pd.DataFrame(issues, columns=['row', 'experiment_id', 'check', 'details'])
optional_missingness = pd.DataFrame([
    {
        'column': column,
        'missing_count': int(df[column].isna().sum()),
        'available_count': int(df[column].notna().sum()),
        'note': '允许缺失，不计为质量错误',
    }
    for column in optional_columns if column in df.columns
])

## 3. 检查结果

In [4]:
display(quality_summary)
passed = int(quality_summary['result'].eq('PASS').sum())
failed = int(quality_summary['result'].eq('FAIL').sum())
print(f'通过检查：{passed} 项；未通过：{failed} 项；异常明细：{len(quality_issues)} 条')

if quality_issues.empty:
    print('结论：30条主数据通过本轮自动质量检查。')
else:
    display(quality_issues)

,check,result,observed,expected,details
0,row_count,PASS,30,30,主数据表应包含30条已核实实验记录。
1,required_columns_present,PASS,13,13,缺失列：无
2,experiment_id_complete,PASS,0,0,每条记录必须有实验编号。
3,experiment_id_unique,PASS,0,0,实验编号必须唯一。
4,required_values_complete,PASS,0,0,必填字段不得为空。
5,numeric_fields_parseable,PASS,0,0,已有的数值字段必须能够转换为数值。
6,core_physical_values_positive,PASS,0,0,沉积时间、压力和膜厚必须大于0。
7,power_or_current_available,PASS,0,0,每条记录至少需要一个正的功率或放电电流值。
8,source_identifiers_valid,PASS,0,0,DOI格式和来源URL必须可识别。
9,verification_status_consistent,PASS,0,0,主表仅允许 verified 且 reported_measured 的记录。


通过检查：10 项；未通过：0 项；异常明细：0 条
结论：30条主数据通过本轮自动质量检查。


## 4. 数据构成与允许缺失字段

In [5]:
print('按材料统计：')
display(df['target_material'].value_counts().rename_axis('material').reset_index(name='records'))

print('按论文来源统计：')
display(df['source_id'].value_counts().sort_index().rename_axis('source_id').reset_index(name='records'))

print('允许缺失的可选字段：')
display(optional_missingness)

按材料统计：


,material,records
0,Cu3N,12
1,Ti,9
2,Pt,8
3,W,1


按论文来源统计：


,source_id,records
0,S01,8
1,S02,12
2,S03,1
3,S05,9


允许缺失的可选字段：


,column,missing_count,available_count,note
0,power_W,8,22,允许缺失，不计为质量错误
1,discharge_current_mA,22,8,允许缺失，不计为质量错误
2,substrate_temperature_C,30,0,允许缺失，不计为质量错误
3,substrate_temperature_label,10,20,允许缺失，不计为质量错误
4,deposition_rate_nm_min,22,8,允许缺失，不计为质量错误
5,original_deposition_time_s,9,21,允许缺失，不计为质量错误
6,comments,0,30,允许缺失，不计为质量错误


## 5. 保存质量检查结果

运行本单元格后，会在主数据CSV旁边的 `quality_checks` 文件夹中保存汇总、异常明细、可选字段缺失统计和JSON报告。

In [6]:
quality_summary.to_csv(output_dir / 'data_quality_summary.csv', index=False, encoding='utf-8-sig')
quality_issues.to_csv(output_dir / 'data_quality_issues.csv', index=False, encoding='utf-8-sig')
optional_missingness.to_csv(output_dir / 'optional_missingness.csv', index=False, encoding='utf-8-sig')

report = {
    'data_file': data_path.name,
    'row_count': int(len(df)),
    'column_count': int(len(df.columns)),
    'passed_checks': int(quality_summary['result'].eq('PASS').sum()),
    'failed_checks': int(quality_summary['result'].eq('FAIL').sum()),
    'issue_count': int(len(quality_issues)),
    'sources': {str(k): int(v) for k, v in df['source_id'].value_counts().sort_index().items()},
    'materials': {str(k): int(v) for k, v in df['target_material'].value_counts().sort_index().items()},
}
(output_dir / 'data_quality_report.json').write_text(
    json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8'
)

print('结果已保存到：', output_dir)
print(json.dumps(report, ensure_ascii=False, indent=2))

结果已保存到： /mnt/d/学业/项目名/outputs/pvd_public_data_v3/quality_checks
{
  "data_file": "pvd_experiments_verified_v3.csv",
  "row_count": 30,
  "column_count": 22,
  "passed_checks": 10,
  "failed_checks": 0,
  "issue_count": 0,
  "sources": {
    "S01": 8,
    "S02": 12,
    "S03": 1,
    "S05": 9
  },
  "materials": {
    "Cu3N": 12,
    "Pt": 8,
    "Ti": 9,
    "W": 1
  }
}


## 本轮结论如何用于后续步骤

只有当所有检查均为 `PASS` 且异常明细为空时，才将这30条记录作为下一份探索性分析Notebook的输入。可选字段缺失代表不同论文采用了不同的实验报告方式，不自动等同于数据错误。